# Baseline model for batch monitoring example

In [4]:
import requests
import datetime
import pandas as pd

from evidently import ColumnMapping
#from evidently.column_mapping import ColumnMapping

from evidently.report import Report
from evidently.metrics import ColumnDriftMetric, DatasetDriftMetric, DatasetMissingValuesMetric

from joblib import load, dump
from tqdm import tqdm

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [5]:
import evidently
print(evidently.__version__)

0.4.18


In [6]:
#files = [('green_tripdata_2022-02.parquet', './data'), ('green_tripdata_2022-01.parquet', './data')]
files = [('green_tripdata_2024-03.parquet', './data')]

print("Download files:")
for file, path in files:
    url=f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file}"
    print(f"Downloading {file} from {url} to {path}")
    resp=requests.get(url, stream=True)
    save_path=f"{path}/{file}"
    with open(save_path, "wb") as handle:
        for data in tqdm(resp.iter_content(),
                        desc=f"{file}",
                        postfix=f"save to {save_path}",
                        total=int(resp.headers["Content-Length"])):
            handle.write(data)
    

Download files:


green_tripdata_2024-03.parquet: 100%|██████████| 919/919 [00:00<00:00, 189339.10it/s, save to ./data/green_tripdata_2024-03.parquet]


In [14]:
!wget https://raw.githubusercontent.com/DataTalksClub/nyc-tlc-data/main/green_tripdata_2024-03.parquet -O data/green_tripdata_2024-03.parquet


--2025-06-24 17:26:09--  https://raw.githubusercontent.com/DataTalksClub/nyc-tlc-data/main/green_tripdata_2024-03.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-06-24 17:26:09 ERROR 404: Not Found.



In [17]:
mar_data = pd.read_parquet('data/green_tripdata_2024-03.parquet')

In [18]:
mar_data.describe()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
count,57457.000000,57457,57457,55360.000000,57457.000000,57457.000000,55360.000000,57457.000000,57457.000000,57457.000000,57457.000000,57457.000000,57457.000000,0.0,57457.000000,57457.000000,55360.000000,55353.000000,55360.000000
mean,1.877334,2024-03-16 04:02:52.405399,2024-03-16 04:21:00.076039,1.179986,95.524688,138.629149,1.309538,13.522828,17.313474,0.904472,0.577410,2.386255,0.192537,NaN,0.979378,22.904832,1.321062,1.038047,0.737730
min,1.000000,2008-12-31 23:02:24,2008-12-31 23:02:30,1.000000,1.000000,1.000000,0.000000,0.000000,-295.080000,-2.500000,-0.500000,-1.560000,0.000000,NaN,-1.000000,-296.080000,1.000000,1.000000,-2.750000
25%,2.000000,2024-03-08 13:53:56,2024-03-08 14:13:49,1.000000,74.000000,74.000000,1.000000,1.100000,9.300000,0.000000,0.500000,0.000000,0.000000,NaN,1.000000,13.440000,1.000000,1.000000,0.000000
50%,2.000000,2024-03-15 22:49:01,2024-03-15 23:09:52,1.000000,75.000000,138.000000,1.000000,1.790000,13.500000,0.000000,0.500000,2.000000,0.000000,NaN,1.000000,18.500000,1.000000,1.000000,0.000000
75%,2.000000,2024-03-23 20:11:25,2024-03-23 20:34:48,1.000000,97.000000,220.000000,1.000000,3.100000,19.800000,1.000000,0.500000,3.610000,0.000000,NaN,1.000000,27.050000,2.000000,1.000000,2.750000
max,2.000000,2024-04-01 00:01:45,2024-04-01 16:11:00,99.000000,265.000000,265.000000,9.000000,125112.200000,841.600000,10.000000,4.250000,150.000000,26.760000,NaN,1.000000,856.980000,5.000000,2.000000,2.750000
std,0.328056,NaN,NaN,1.356719,57.285088,76.295346,0.967749,770.416255,14.958249,1.382446,0.366916,3.159273,1.184551,NaN,0.154253,17.013735,0.497858,0.191311,1.218039


In [19]:
mar_data.shape

(57457, 20)

In [26]:
# create target
mar_data["duration_min"] = mar_data.lpep_dropoff_datetime - mar_data.lpep_pickup_datetime
mar_data.duration_min = mar_data.duration_min.apply(lambda td : float(td.total_seconds())/60)

In [27]:
# filter out outliers
mar_data = mar_data[(mar_data.duration_min >= 0) & (mar_data.duration_min <= 60)]
mar_data = mar_data[(mar_data.passenger_count > 0) & (mar_data.passenger_count <= 8)]

In [29]:
# data labeling
target = "duration_min"
num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

In [49]:
train_data = mar_data[:30000]
val_data = mar_data[30000:]

In [50]:
model = LinearRegression()

In [51]:
model.fit(train_data[num_features + cat_features], train_data[target])

LinearRegression()

In [52]:
train_preds = model.predict(train_data[num_features + cat_features])
train_data['prediction'] = train_preds

In [53]:
val_preds = model.predict(val_data[num_features + cat_features])
val_data['prediction'] = val_preds

In [54]:
print(mean_absolute_error(train_data.duration_min, train_data.prediction))
print(mean_absolute_error(val_data.duration_min, val_data.prediction))

3.7724732393594484
3.716814567929369


# Dump model and reference data

In [55]:
with open('models/lin_reg.bin', 'wb') as f_out:
    dump(model, f_out)

In [56]:
val_data.to_parquet('data/reference.parquet')

# Evidently Report

In [57]:
column_mapping = ColumnMapping(
    target=None,
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features
)

In [58]:
from evidently.metrics import ColumnQuantileMetric, DatasetSummaryMetric

# Add a data quality metric (DatasetSummaryMetric) and a quantile metric for "fare_amount"
report = Report(metrics=[
    ColumnDriftMetric(column_name='prediction'),
    DatasetDriftMetric(),
    DatasetMissingValuesMetric(),
    DatasetSummaryMetric(),
    ColumnQuantileMetric(column_name="fare_amount", quantile=0.5)
])


In [59]:
report.run(reference_data=train_data, current_data=val_data, column_mapping=column_mapping)

In [60]:
report.show(mode='inline')

In [61]:
result = report.as_dict()

In [62]:
result

{'metrics': [{'metric': 'ColumnDriftMetric',
   'result': {'column_name': 'prediction',
    'column_type': 'num',
    'stattest_name': 'Wasserstein distance (normed)',
    'stattest_threshold': 0.1,
    'drift_score': 0.010064593780096743,
    'drift_detected': False,
    'current': {'small_distribution': {'x': [-103.3888256009412,
       -78.94485371737699,
       -54.500881833812784,
       -30.05690995024858,
       -5.6129380666843645,
       18.831033816879852,
       43.27500570044404,
       67.71897758400826,
       92.16294946757247,
       116.60692135113669,
       141.0508932347009],
      'y': [1.6950437862191267e-06,
       0.0,
       0.0,
       5.08513135865738e-06,
       0.03510435681259812,
       0.005500417086281073,
       0.0002610367430777455,
       3.051078815194428e-05,
       3.3900875724382533e-06,
       3.3900875724382533e-06]}},
    'reference': {'small_distribution': {'x': [-57.57949648545787,
       -41.48546822381036,
       -25.391439962162842,
    

In [67]:
daily_median = mar_data.set_index('lpep_pickup_datetime').resample('D')['fare_amount'].quantile(0.5)
# Filter for March 2024 before calculating the max median
march_median = daily_median[daily_median.index.month == 3]
max_daily_median = march_median.max()
max_daily_median

14.2

# Q4 Evidently Dashboard

In [75]:
from evidently.metric_preset import DataDriftPreset, DataQualityPreset

from evidently.ui.workspace import Workspace
from evidently.ui.dashboards import DashboardPanelCounter, DashboardPanelPlot, CounterAgg, PanelValue, PlotType, ReportFilter
from evidently.renderers.html_widgets import WidgetSize

In [76]:
ws = Workspace("workspace")

In [90]:
project = ws.create_project("NYC Green Taxi (Mar 2024) Data Quality Project")
project.description = "My homework 05-monitoring"
project.save()

Project(id=UUID('01c2a839-6786-4953-8ab9-03a2a10d5f73'), name='NYC Green Taxi (Mar 2024) Data Quality Project', description='My homework 05-monitoring', dashboard=DashboardConfig(name='NYC Green Taxi (Mar 2024) Data Quality Project', panels=[], tabs=[], tab_id_to_panel_ids={}), team_id=None, date_from=None, date_to=None)

In [95]:
import os

# Add new panels for the additional metrics to the dashboard
project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="Prediction Drift Score",
        values=[
            PanelValue(
                metric_id="ColumnDriftMetric",
                field_path="drift_score",
                legend="Prediction Drift"
            ),
        ],
        plot_type=PlotType.LINE,
        size=WidgetSize.HALF,
    ),
)

project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="Number of Drifted Columns",
        values=[
            PanelValue(
                metric_id="DatasetDriftMetric",
                field_path="number_of_drifted_columns",
                legend="Drifted Columns"
            ),
        ],
        plot_type=PlotType.BAR,
        size=WidgetSize.HALF,
    ),
)

project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="Share of Missing Values",
        values=[
            PanelValue(
                metric_id="DatasetMissingValuesMetric",
                field_path="current.share_of_missing_values",
                legend="Missing Values Share"
            ),
        ],
        plot_type=PlotType.LINE,
        size=WidgetSize.HALF,
    ),
)

project.dashboard.add_panel(
    DashboardPanelPlot(
        filter=ReportFilter(metadata_values={}, tag_values=[]),
        title="Number of Missing Values",
        values=[
            PanelValue(
                metric_id="DatasetSummaryMetric",
                field_path="current.number_of_missing_values",
                legend="count"
            ),
        ],
        plot_type=PlotType.LINE,
        size=WidgetSize.HALF,
    ),
)

title='Number of Missing Values',
values=[
    PanelValue(
        metric_id="DatasetSummaryMetric",
        field_path="current.number_of_missing_values",
        legend="count"
    ),
],
plot_type=PlotType.LINE,
size=WidgetSize.HALF,
project.save()

# Save dashboard configuration as JSON
dashboard_config_path = "dashboards/dashboard_config.json"
os.makedirs(os.path.dirname(dashboard_config_path), exist_ok=True)
with open(dashboard_config_path, "w") as f:
    f.write(project.dashboard.json())

In [96]:
regular_report = Report(
    metrics=[
        DataQualityPreset()
    ],
    timestamp=datetime.datetime(2022,1,28)
)

regular_report.run(reference_data=None,
                  current_data=val_data.loc[val_data.lpep_pickup_datetime.between('2024-02-28', '2024-03-31', inclusive="left")],
                  column_mapping=column_mapping)

regular_report